RAG implementation starting with 
data ingestion pipeline 

In [2]:
from langchain_community.document_loaders import DirectoryLoader,PyPDFLoader


In [ ]:
loader=DirectoryLoader(
    "../data",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)
mydocs=loader.load()
type_counts={}

for doc in mydocs:
    source=doc.metadata.get("source","").lower()

    if "curriculum" in source:
       doc.metadata["type"]="course"
    elif "student" in source:
       doc.metadata["type"]="byLaw"
    elif "regulation" in source:
       doc.metadata["type"]="regulation"
    else:
       doc.metadata["type"]="general"   

    print("\n..new document..")
    print("source",source)
    print("type",doc.metadata["type"])

    t = doc.metadata["type"]
    type_counts[t] = type_counts.get(t, 0) + 1

print("\nSUMMARY:")
print(type_counts)
               
#texts
# for i,doc in enumerate(mydocs):
#     print(f"document {i+1}\n")
#     print(f"cont:{doc.page_content}\n")
#     print(f"metadata:{doc.metadata}\n")
    
##mydocs



..new document..
source ..\data\regulation_for_undegraduate.pdf
type regulation

..new document..
source ..\data\regulation_for_undegraduate.pdf
type regulation

..new document..
source ..\data\regulation_for_undegraduate.pdf
type regulation

..new document..
source ..\data\regulation_for_undegraduate.pdf
type regulation

..new document..
source ..\data\regulation_for_undegraduate.pdf
type regulation

..new document..
source ..\data\regulation_for_undegraduate.pdf
type regulation

..new document..
source ..\data\regulation_for_undegraduate.pdf
type regulation

..new document..
source ..\data\regulation_for_undegraduate.pdf
type regulation

..new document..
source ..\data\regulation_for_undegraduate.pdf
type regulation

..new document..
source ..\data\regulation_for_undegraduate.pdf
type regulation

..new document..
source ..\data\regulation_for_undegraduate.pdf
type regulation

..new document..
source ..\data\regulation_for_undegraduate.pdf
type regulation

..new document..
source ..\

In [3]:
from langchain_core.documents import Document
import re

In [ ]:
from importlib import metadata

from sympy import Line

# import re
# from langchain_core.documents import Document

def parsed_curriculum(text):
    if not isinstance(text, str):
        return []

    curriculum_doc = []

    current_college = None
    current_program = None
    current_year = None
    current_semester = None

    lines = text.split("\n")

    for line in lines:
        line = line.strip()
        if not line:
            continue

       
        # Clean numbering first
        line = re.sub(r"^\d+(\.\d+)*\s*", "", line)

        lower = line.lower()
        
        
        college_match = re.search(r"(college of [a-z\s&]+)", lower)
        if college_match:
            cleaned = college_match.group(1)
            cleaned = re.sub(r"\(.*?\)", "", cleaned)   # remove parentheses
            cleaned = re.sub(r"\.+\s*\d*", "", cleaned) # remove trailing dots/numbers

            # Normalize spacing and remove trailing "offers..."
            cleaned = re.sub(r"\s+", " ", cleaned)
            cleaned = re.sub(r"offers.*", "", cleaned)

            current_college = cleaned.strip().lower()
            #print("SET COLLEGE:", current_college)
            continue

         


        #Program 
        if "bachelor of" in lower:
           program_match = re.match(r"^(bachelor of [^,:\n]+)", lower)
           if program_match:
                current_program = program_match.group(1).strip()
                current_year = None
                current_semester = None
               
                # if current_college is None:
                #     print(" Program found before college!")
           continue

        # Year
        year_match = re.search(r"year\s+(one|two|three|four)", lower)
        if year_match:
            current_year = year_match.group(1)
            continue

        # Semester
        semester_match = re.search(r"semester\s+(one|two)", lower)
        if semester_match:
            current_semester = semester_match.group(1)
            continue

        
        # Course detection
        if current_program and current_year and current_semester:
            # Match course codes like CSC 101, EDU 1101, MATH2100
            if re.search(r"[A-Z]{2,}\s*\d{2,4}", line):
                curriculum_doc.append(Document(
                    page_content=line,
                    metadata={
                        "type": "course",
                        "college": current_college if current_college else None,
                        "program": current_program,
                        "year": current_year,
                        "semester": current_semester
                    }
                ))


    return curriculum_doc






   


In [5]:
from pydoc import text
from re import search


def parsed_regulation(text):
    regulation_docs=[]

    sections=re.split(r"\n(?=\d+\.\d+)",text)
    print("sections broken successfully")

    for sec in sections:
        sec=sec.strip()

        if not sec:
            continue

        section_match=re.search(r"(\d+\.\d+)",sec)
        
        if not section_match:
            continue 
        section_number=section_match.group(1)
         
        print(section_number)

        def topic_tagging(text):
            text=text.lower()
            if "gpa" in text:
                return "gpa"
            if "carry" in text:
                return "carry"
            if "credit" in text:
                return "credit"
            if "classification" in text:
                return "degree_classification"
            return "general"

        regulation_docs.append(Document(
            page_content=sec,
            metadata={
                "type":"regulation",
                 "section":section_number,
                 "topic":topic_tagging(sec)

            }
        )
        )
    return regulation_docs
    




        







In [6]:
def parsed_bylaw(text):
    bylaw_docs=[]
    lines=text.split("\n")
    current_section=None
    buffer=""

  
    for line in lines:
        line=line.strip()
        if not line:
            continue
        lower=line.lower()

        sec_match=re.search(r"^\d+(\.\d+)*",line)

        if sec_match:
            current_section=sec_match.group(0)
            #print(current_section)
            continue

        bullet_match=re.search(r"^\(?[ivx]+\)",lower)
        if bullet_match:
            if buffer:
                bylaw_docs.append(Document(
                    page_content=buffer.strip(),
                    metadata={
                        "type":"byLaw",
                        "section":current_section

                    }


                ))
                buffer=""
            buffer=line
            continue

        buffer+=" "+line  
    if buffer:
        bylaw_docs.append(Document(
            page_content=buffer.strip(),
            metadata={
                "type":"byLaw",
                "section":current_section
            }
        ))  
    return bylaw_docs        



       






In [7]:
all_documents=[]
curriculum_text=""
regulation_text=""
byLaw_text=""
for doc in mydocs:
    source=doc.metadata.get("source","").lower()

    if source.endswith("curriculum_guidebook_2021_2022.pdf"):
        curriculum_text+=doc.page_content+ "\n"

    elif "regulation" in source:
        regulation_text+=doc.page_content+ "\n"   
    elif "student" in source:
        byLaw_text+=doc.page_content+"\n"     
        
curriculum_docs=parsed_curriculum(curriculum_text)
regulation_docs=parsed_regulation(regulation_text)
byLaw_docs=parsed_bylaw(byLaw_text)

all_documents.extend(curriculum_docs)
all_documents.extend(regulation_docs)
all_documents.extend(byLaw_docs)


print("Curriculum docs:", len(curriculum_docs))
print("Regulation docs:", len(regulation_docs))
print("bylaw docs:",len(byLaw_docs))
print("Total parsed docs:", len(all_documents))

for d in all_documents[0:1005]:
    print("\n---")
    print("Metadata:", d.metadata)
    print("Content:", d.page_content)


SET COLLEGE: college of business and economics
SET COLLEGE: college of earth sciences and engineering
SET COLLEGE: college of education
SET COLLEGE: college of humanities and social sciences
SET COLLEGE: college of informatics and virtual education
SET COLLEGE: college of natural and mathematical sciences
SET COLLEGE: college of business and economics
SET COLLEGE: college of business and economics
SET COLLEGE: college of earth sciences and engineering
SET COLLEGE: college of earth sciences
SET COLLEGE: college of education
SET COLLEGE: college of education is to train high
SET COLLEGE: college of education
SET COLLEGE: college of humanities and social sciences
SET COLLEGE: college of humanities and social sciences
SET COLLEGE: college of informatics and virtual education
SET COLLEGE: college of informatics and virtual education
SET COLLEGE: college of natural and mathematical sciences
SET COLLEGE: college of natural and mathematical sciences
SET COLLEGE: college of natural and mathemat

In [10]:
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
import os

In [12]:
import shutil
from collections import Counter
from langchain_openai import OpenAIEmbeddings

embeddings=OpenAIEmbeddings(model="text-embedding-3-small")
# project_root=os.path.dirname(os.getcwd)
# persist_directory=os.path.join(project_root,"chroma_db")
persist_directory="../chroma_db"

shutil.rmtree("./chroma_db", ignore_errors=True)

stored_vector=Chroma.from_documents(
    embedding=embeddings,
    documents=all_documents,
    persist_directory=persist_directory,
    collection_name="university_documents"


)
print(f"documents stored succesfully , persisted here:{persist_directory}")

all_docs = stored_vector.get()

types = [m["type"] for m in all_docs["metadatas"]]


print(Counter(types))

documents stored succesfully , persisted here:../chroma_db
Counter({'course': 3560, 'regulation': 381, 'byLaw': 100})


RETRIEVAL PIPELINE

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [21]:
retriever=stored_vector.as_retriever(search_kwargs={"k":3})

query=retriever.invoke("are students required to open bank account")

# for i,j in enumerate(query):
#     print(f"result{i+1}\n {doc.page_content}")
#     print(f"metadata:{doc.metadata["type"]}")
print(query)

[Document(id='2e39e555-7abc-40b7-9621-84c4dc08dd26', metadata={'section': '20', 'type': 'byLaw'}, page_content='(iv) All students are advised  to open a Bank account with any Bank in Dodoma. UDOM Students By-Laws'), Document(id='314e03a1-89f8-4fc6-a260-7cb35b623dbf', metadata={'type': 'regulation', 'section': '39.6', 'topic': 'general'}, page_content='39.6 Every student shall have the duty to observe the following in respect of UDOM \nSR: \ni. To keep confidential his or her account credentials and prevent an \nunauthorized person from accessing or making an alteration to any \nsuch details, which are within the control of the account holder;  \nii. To make a follow up of his or her studentship academic performance \nstatus throughout the period of his/her study, and \niii. Reporting to the Pri ncipal/Dean/Director of a teaching unit any \nanomaly or unwanted details in his or her UDOM SR account.\n38'), Document(id='c1b7e834-990b-4af7-8be4-4b5db8580c05', metadata={'topic': 'general', 